# foldgate quickstart

Turn a frozen co-folding model's confidence into a **risk-controlled accept/abstain** decision with a finite-sample conformal guarantee, then show why the guarantee needs the shift-robust variant on novel targets.

Prereq: `python scripts/download_data.py && make features` (produces `data/processed/rnp_delivered.parquet`). This notebook is torch-free and runs on CPU in seconds.

In [ ]:
import sys; sys.path.insert(0, 'src')
import numpy as np, pandas as pd
from foldgate.scores import ScoreCombiner
from foldgate.conformal import ltt_threshold
from foldgate.selective import evaluate_gate, aurc

df = pd.read_parquet('data/processed/rnp_delivered.parquet')
sub = df[df.method == 'af3'].reset_index(drop=True)
y = sub['correct'].to_numpy()            # 1 iff BiSyRMSD <= 2 A
rng = np.random.default_rng(0)
print(f'{len(sub)} AF3 delivered poses, base correct = {y.mean():.3f}')

## 1. i.i.d. gate with a guarantee

Fit the combiner on TRAIN, certify a Learn-then-Test threshold on CAL (`alpha=0.2` error budget, `delta=0.1` failure prob), evaluate on TEST. The gate accepts the poses it can certify at <= 20% error and abstains on the rest.

In [ ]:
ALPHA, DELTA = 0.20, 0.10
idx = rng.permutation(len(sub)); a, b = int(.4*len(sub)), int(.7*len(sub))
tr, cal, te = idx[:a], idx[a:b], idx[b:]
comb = ScoreCombiner().fit(sub.iloc[tr], y[tr])
sc_cal, sc_te = comb.predict(sub.iloc[cal]), comb.predict(sub.iloc[te])
tau = ltt_threshold(sc_cal, y[cal], alpha=ALPHA, delta=DELTA)
g = evaluate_gate(sc_te, y[te], tau)
print(f"accepted {g['coverage']:.0%} of poses at realized error {g['selective_risk']:.3f} (target {ALPHA})")
print(f"combined-score AURC {aurc(sc_te, y[te]):.3f}  vs  native ranking_score AURC "
      f"{aurc(sub['ranking_score'].to_numpy()[te], y[te]):.3f}  (lower = better)")

## 2. The guarantee breaks under novelty shift

Calibrate on the most familiar targets (novelty stratum S0) and deploy on progressively novel ones (S1-S3). Exchangeability is gone, so the transferred threshold silently under-controls error.

In [ ]:
strat = sub['novelty_stratum'].to_numpy().astype(int)
src = np.where(strat == 0)[0]; tgt = np.where(np.isin(strat, [1,2,3]))[0]
sp = rng.permutation(src); s_tr, s_cal = sp[:len(src)//2], sp[len(src)//2:]
comb_s = ScoreCombiner().fit(sub.iloc[s_tr], y[s_tr])
tau_src = ltt_threshold(comb_s.predict(sub.iloc[s_cal]), y[s_cal], alpha=ALPHA, delta=DELTA)
gb = evaluate_gate(comb_s.predict(sub.iloc[tgt]), y[tgt], tau_src)
print(f"naive transfer to novel targets: accepted {gb['coverage']:.0%} at realized error "
      f"{gb['selective_risk']:.3f}  >>  target {ALPHA}  (the exchangeability break)")

## 3. Repair it with group-conditional (Mondrian) conformal

Calibrate a separate threshold per novelty stratum using target-stratum labels. Each stratum then respects the error budget, at an honest coverage cost where the model is unreliable. (When you cannot label the target, use `weighted_ltt_threshold` instead; see E3b.)

In [ ]:
tp = rng.permutation(tgt); t_cal, t_test = tp[:len(tgt)//2], tp[len(tgt)//2:]
sc_tc, sc_tt = comb_s.predict(sub.iloc[t_cal]), comb_s.predict(sub.iloc[t_test])
n_acc = err = 0
for k in np.unique(strat[t_test]):
    ck = strat[t_cal] == k
    if ck.sum() < 40: continue
    tk_tau = ltt_threshold(sc_tc[ck], y[t_cal][ck], alpha=ALPHA, delta=DELTA)
    if tk_tau is None: continue
    tk = strat[t_test] == k; acc = sc_tt[tk] >= tk_tau
    n_acc += int(acc.sum()); err += int((1 - y[t_test][tk][acc]).sum())
risk = err/n_acc if n_acc else float('nan')
print(f"group-conditional on novel targets: accepted {n_acc/len(t_test):.0%} at realized error "
      f"{risk:.3f}  <=  target {ALPHA}  (validity restored)")

## Takeaway

- The combined score is a better reliability ranker than native confidence (lower AURC).
- Split conformal gives a finite-sample coverage guarantee i.i.d., but it **breaks under novel-pocket / novel-chemotype shift**.
- Group-conditional (labels available) or weighted (label-free) conformal keyed on training-set similarity **repairs** the guarantee. Ordinary calibration has no such repair.